In [ ]:
!pip install transformers datasets sentencepiece -q


In [ ]:
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments


Load dataset

In [ ]:
dataset = load_dataset("glue", "sst2")

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["sentence"], padding=True, truncation=True)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


Load model & train

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
)

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"]
)

trainer.train()


text corpus

In [ ]:
texts = [
    "Natural language processing is interesting",
    "BERT models use subword tokenization",
    "SentencePiece is language independent"
]

with open("corpus.txt", "w") as f:
    for line in texts:
        f.write(line + "\n")


Train SentencePiece

In [ ]:
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input="corpus.txt",
    model_prefix="spm_model",
    vocab_size=200
)


Load tokenizer

In [ ]:
sp = spm.SentencePieceProcessor()
sp.load("spm_model.model")

print(sp.encode("BERT uses subword tokens", out_type=str))


Pipeline-based inference

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="bert-base-uncased-finetuned-sst-2-english"
)

classifier("This movie was absolutely fantastic!")


Manual inference

In [ ]:
inputs = tokenizer("The movie was boring", return_tensors="pt")
outputs = model(**inputs)
pred = outputs.logits.argmax(dim=1)
print(pred.item())
